1. Carregar dados do CSV AIRPLANE (FEITO)
2. Tratar esses dados (FEITO)
3. Jogar em um banco (Engine, SQL Script) (FEITO)

In [21]:
import pandas as pd
import numpy as np
import psycopg2
from dotenv import load_dotenv
import os


In [22]:
load_dotenv()

True

In [23]:
db_user = os.getenv("USER_POSTGRE")
db_password = os.getenv("DB_PASSWORD")

# Extract

In [24]:
df_airplany = pd.read_csv('../csv/2007.csv')

In [25]:
df_airplany

,Year,Month,DayofMonth,DayOfWeek,DepTime,CRSDepTime,ArrTime,CRSArrTime,UniqueCarrier,FlightNum,...,TaxiIn,TaxiOut,Cancelled,CancellationCode,Diverted,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay
0,2007,1,1,1,1232.0,1225,1341.0,1340,WN,2891,...,4,11,0,NaN,0,0,0,0,0,0
1,2007,1,1,1,1918.0,1905,2043.0,2035,WN,462,...,5,6,0,NaN,0,0,0,0,0,0
2,2007,1,1,1,2206.0,2130,2334.0,2300,WN,1229,...,6,9,0,NaN,0,3,0,0,0,31
3,2007,1,1,1,1230.0,1200,1356.0,1330,WN,1355,...,3,8,0,NaN,0,23,0,0,0,3
4,2007,1,1,1,831.0,830,957.0,1000,WN,2278,...,3,9,0,NaN,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7453210,2007,12,15,6,1558.0,1605,1749.0,1736,DL,58,...,14,17,0,NaN,0,0,0,0,0,0
7453211,2007,12,15,6,1902.0,1851,2110.0,2105,DL,59,...,6,21,0,NaN,0,0,0,0,0,0
7453212,2007,12,15,6,1024.0,1025,1750.0,1735,DL,61,...,14,19,0,NaN,0,0,0,15,0,0
7453213,2007,12,15,6,1353.0,1315,1658.0,1622,DL,62,...,11,14,0,NaN,0,0,0,0,0,36


# Transform

In [26]:
df_airplany = df_airplany.rename(columns={
    'Year':'year','Month':'month','DayOfMonth':'day_of_month','DayOfWeek':'day_of_week','DepTime':'dep_time','CRSDepTime':'crs_dep_time', 
    'ArrTime':'arr_time', 'CRSArrTime':'crs_arr_time', 'UniqueCarrier':'unique_carrier', 'FlightNum':'flight_num', 'TaxiIn':'taxin_in', 'TaxiOut':'taxi_out', 
    'Cancelled':'cancelled', 'CancellationCode':'cancellation_code', 'Diverted':'diverted','CarrierDelay':'carrier_delay', 'WeatherDelay':'weater_delay',
    'NASDelay':'nas_delay',	'SecurityDelay':'security_delay', 'LateAircraftDelay':'late_aircraft_delay' 
}).drop_duplicates()

In [27]:
df_airplany = df_airplany.replace({np.nan: None})

In [28]:
print(df_airplany.dtypes)

year                    int64
month                   int64
DayofMonth              int64
day_of_week             int64
dep_time               object
crs_dep_time            int64
arr_time               object
crs_arr_time            int64
unique_carrier            str
flight_num              int64
TailNum                object
ActualElapsedTime      object
CRSElapsedTime         object
AirTime                object
ArrDelay               object
DepDelay               object
Origin                    str
Dest                      str
Distance                int64
taxin_in                int64
taxi_out                int64
cancelled               int64
cancellation_code      object
diverted                int64
carrier_delay           int64
weater_delay            int64
nas_delay               int64
security_delay          int64
late_aircraft_delay     int64
dtype: object


# Load

- ENGINE XXXXX ERROR XXXXX
- SCRIPT.SQL XXXXX ERROR XXXXX

In [29]:
df_airplany.to_csv("voos.csv", index=False)

In [30]:
conexao = psycopg2.connect(host="localhost",database="manu_tasks",user=db_user,password=db_password,port=5432)

cursor = conexao.cursor()

In [31]:
cursor.execute("TRUNCATE TABLE public.voos;")

In [32]:
with open("../Manu/voos.csv", "r", encoding="utf-8") as arquivo:
    cursor.copy_expert(
        """ COPY public.voos (
            year,month,day_of_month,day_of_week,dep_time,crs_dep_time,
            arr_time,crs_arr_time,unique_carrier,flight_num,tail_num,actual_elapsed_time,
            crs_elapsed_time,air_time,arr_delay,dep_delay,origin,dest,distance,
            taxin_in,taxi_out,cancelled,cancellation_code,diverted,carrier_delay,
            weater_delay,nas_delay,security_delay,late_aircraft_delay
        )FROM STDIN WITH CSV HEADER """,
        arquivo
    )

conexao.commit()